# Train NN models with Azure Machine Learning

*The example code in this notebook trains a TensorFlow model to classify bank churn for a bank, and create an endpoint for predictions.*

## Connect to the workspace

*First, we need to connect to our Azure Machine Learning workspace. The Azure Machine Learning workspace is the top-level resource for the service. It provides us with a centralized place to work with all the artifacts we create when we use Azure Machine Learning.*

*We're using DefaultAzureCredential to get access to the workspace. This credential should be capable of handling most Azure SDK authentication scenarios.*

In [2]:
# Handle to the workspace
from azure.ai.ml import MLClient

# Authentication package
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()

from azure.ai.ml import command, Input

from azure.ai.ml.entities import (
    ManagedOnlineEndpoint,
    ManagedOnlineDeployment,
    Model,
    CodeConfiguration
)

from azure.storage.blob import BlobServiceClient

*Next, get a handle to the workspace by providing your Subscription ID, Resource Group name, and workspace name. To find these parameters:*

* Look in the upper-right corner of the Azure Machine Learning studio toolbar for your workspace name.
* Select your workspace name to show your Resource Group and Subscription ID.
* Copy the values for Resource Group and Subscription ID into the code.

In [3]:
# Get a handle to the workspace
ml_client = MLClient(
    credential=credential,
    subscription_id="6c23abb4-31cd-4bb7-bd34-4f4d970f93ea", #Provide your subscription ID as shown in the above screenshot
    resource_group_name="gl-oct24-pro", #Provide your Resource Group as shown in the above screenshot
    workspace_name="gl-oct24-pro",
)

*The result of running this script is a workspace handle that we use to manage other resources and jobs.*

## Create a compute resource

*Azure Machine Learning needs a compute resource to run a job. This resource can be single or multi-node machines with Linux or Windows OS, or a specific compute fabric like Spark.*

*In the following example script, we provision a Linux compute cluster. You can see the Azure Machine Learning pricing page --> "https://azure.microsoft.com/en-gb/pricing/details/machine-learning/" for the full list of VM sizes and prices. Since we need a GPU cluster for this example, let's pick a **Standard_NC4as_T4_v3** with **4 cores, 28GB RAM, 176GB storage** and create an Azure Machine Learning compute.*

In [4]:
from azure.ai.ml.entities import AmlCompute

gpu_compute_target = "gpu-cluster"

try:
    # let's see if the compute target already exists
    gpu_cluster = ml_client.compute.get(gpu_compute_target)
    print(
        f"You already have a cluster named {gpu_compute_target}, we'll reuse it as is."
    )

except Exception:
    print("Creating a new gpu compute target...")

    # Let's create the Azure ML compute object with the intended parameters
    gpu_cluster = AmlCompute(
        # Name assigned to the compute cluster
        name="gpu-cluster",
        # Azure ML Compute is the on-demand VM service
        type="amlcompute",
        # VM Family
        size="Standard_NC4as_T4_v3",
        # Minimum running nodes when there is no job running
        min_instances=0,
        # Nodes in cluster
        max_instances=4,
        # How many seconds will the node running after the job termination
        idle_time_before_scale_down=180,
        # Dedicated or LowPriority. The latter is cheaper but there is a chance of job termination
        tier="Dedicated",
    )

    # Now, we pass the object to MLClient's create_or_update method
    gpu_cluster = ml_client.begin_create_or_update(gpu_cluster).result()

print(
    f"AMLCompute with name {gpu_cluster.name} is created, the compute size is {gpu_cluster.size}"
)

Creating a new gpu compute target...
AMLCompute with name gpu-cluster is created, the compute size is Standard_NC4as_T4_v3


## Create a job environment

*To run an Azure Machine Learning job, we need an environment. An Azure Machine Learning environment encapsulates the dependencies (such as software runtime and libraries) needed to run our machine learning training script on our compute resource. This environment is similar to a Python environment on our local machine.*

*Azure Machine Learning allows us to either use a curated (or ready-made) environment—useful for common training and inference scenarios—or create a custom environment using a Docker image or a Conda configuration.*

*In this article, we will reuse the curated Azure Machine Learning environment **AzureML-tensorflow-2.7-ubuntu20.04-py38-cuda11-gpu**. We will use the latest version of this environment using the **@latest** directive. - we will add a sklearn component to the data so we can use sklearn*

In [14]:
curated_env_name = "tf_sklearn@latest"

## Configure and submit your training job

*In this section, we begin by introducing the data for training. We then cover how to run a training job, using a training script that we've provided. We learn to build the training job by configuring the command for running the training script. Then, we submit the training job to run in Azure Machine Learning.*

**Obtain the training data**

**We'll use tabular data - a csv file that has the information of different clients and their behavior that could lead to attrition - a synonym of churn**

The variables in the dataset are:
- CLIENTNUM - unused, given that it is an identifier
- Attrition_Flag - The label variable
- Customer_Age
- Gender
- Dependent_count
- Education_level
- Marital_Status
- Income_Category
- Card_Category
- Months_on_book
- Total_Relationship_Count
- Months_Inactive_12_mon
- Contacts_Count_12_mon
- Credit_Limit
- Total_Revolving_Bal
- Avg_Open_To_Buy
- Total_Amt_Chng_Q4_Q1
- Total_Trans_Amt	
- Total_Trans_Ct
- Total_Ct_Chng_Q4_Q1	
- Avg_Utilization_Ratio


## Prepare the training script

*The provided training script does the following:*

1. handles the data preprocessing, splitting the data into test and train data
2. trains a model, using the data
3. returns the output model.


*In the training script **tf_bank.py**, we create a **simple sequential NN**. This script trains a **NN** on the **Bank Dataset**, saves the trained model, and logs performance metrics to Azure ML.*

1. **Configuration:**
Sets up hyperparameters: batch size (32), number of epochs (5), and learning rate (0.001).
2. **Get Current Run:**
Retrieves the current Azure ML run context to log metrics and artifacts.
3. **Load and Preprocess Data:**
Loads data
OneHotEncoder and StandardScaler of categorical and numerical features respectively
4. **Create NN Model:**
Defines a custom NN model.
Layers:
Three Sequential layers with ReLU activation.
Final Output layer with Sigmoid optimizer
5. **Compile Model:**
Configures the model with Adam optimizer, binary_crossentropy, and accuracy metric.
6. **Train Model:**
Fits the model on the training data for the specified number of epochs.
Uses validation data to evaluate the model's performance during training.
7. **Save Model:**
Saves the trained model in the TensorFlow SavedModel format in the outputs directory.
8. **Log Metrics:**
Logs training metrics (loss, accuracy) for each epoch to Azure ML.
9. **Complete Run:**
Marks the end of the Azure ML run.


In [5]:
%%writefile tf-bank.py
# Import necessary libraries
import argparse
import pandas as pd
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Main function to train the model
def main():
    parser = argparse.ArgumentParser()

    # Define command line arguments for training and validation data paths
    parser.add_argument("--data", type=str, required=True, help="Path to data")

    args = parser.parse_args()
    bank = pd.read_csv(args.data)
    bank['Attrition_Flag'] = bank['Attrition_Flag'].apply(lambda x: 0 if "Existing Customer" else 1)
    X = bank.drop(columns=["Attrition_Flag"])
    X = pd.get_dummies(X,drop_first=True)
    y = bank['Attrition_Flag']

    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
    )

    # Scale the data
    S=MinMaxScaler()
    X_train = S.fit_transform(X_train)
    X_test = S.transform(X_test)

    #Initialising the CNN
    nn = tf.keras.models.Sequential()

    #Adding the layers
    nn.add(tf.keras.layers.Dense(units=32, activation='relu'))
    nn.add(tf.keras.layers.Dense(units=16, activation='relu'))
    nn.add(tf.keras.layers.Dense(units=8, activation='relu'))
    nn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))
    nn.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

    # Train the model
    nn.fit(X_train, y_train, batch_size = 32, epochs = 5)

    # Evaluate the model on validation data
    loss, accuracy = nn.evaluate(X_test)
    print(f'Validation Loss: {loss:.4f}')
    print(f'Validation Accuracy: {accuracy:.4f}')

    # Ensure the outputs directory exists to save the model
    os.makedirs("outputs", exist_ok=True)

    # # Export the trained model to the output directory
    # tf.saved_model.save(model, "outputs/smoker_detection_model")
    nn.save("outputs/model.keras")

# Run the main function
if __name__ == '__main__':
    main()

Writing tf-bank.py


## Configure the command

*We use the general purpose command to run the training script and perform our desired tasks. We create a Command object to specify the configuration details of our training job.*

*For the parameter values:*

1. provide the compute cluster **gpu_compute_target** = **"gpu-cluster"** that we created for running this command;
2. provide the curated environment **curated_env_name** that we declared earlier;
3. configure the command line action itself—in this case, the command is **python tf_bank.py**.
4. configure metadata such as the **display name** and **experiment name**; where an experiment is a container for all the iterations one does on a certain project. All the jobs submitted under the same experiment name would be listed next to each other in Azure Machine Learning studio.

In [15]:
from azure.ai.ml import command
from azure.ai.ml import UserIdentityConfiguration
from azure.ai.ml import Input

# Define the job configuration
job = command(
    inputs=dict(
    data=Input(
        type="uri_file",
        path="Bank_Customer_Churn.csv",  # Adjusted to the correct path
    )),
    compute=gpu_compute_target,
    environment=curated_env_name,
    code=".",
    command="python tf-bank.py --data ${{inputs.data}}",
    experiment_name="tf-bank",
    display_name="tensorflow-classify-bank",
)

## Submit the job

*It's now time to submit the job to run in Azure Machine Learning. This time, we'll use create_or_update on ml_client.jobs.*

*Once completed, the job will register a model in our workspace (as a result of training) and output a link for viewing the job in Azure Machine Learning studio.*

In [18]:
training_job = ml_client.jobs.create_or_update(job)

## What happens during job execution

*As the job is executed, it goes through the following stages:*

1. Preparing: A docker image is created according to the environment defined. The image is uploaded to the workspace's container registry and cached for later runs. Logs are also streamed to the job history and can be viewed to monitor progress. If a curated environment is specified, the cached image backing that curated environment will be used.

2. Scaling: The cluster attempts to scale up if it requires more nodes to execute the run than are currently available.

3. Running: All scripts in the script folder src are uploaded to the compute target, data stores are mounted or copied, and the script is executed. Outputs from stdout and the ./logs folder are streamed to the job history and can be used to monitor the job.

## Metrics Overview

*We can check the metrics by looking at the logs - we could get more visual representation if adding ML flow*

## Register the Model

*Register the model through the UI itself instead of code. Follow the below steps to register the model.*

In [20]:
training_job = ml_client.jobs.get(name=training_job.name)

from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes

# Registering the model after training
model = Model(
    path="azureml://jobs/{}/outputs/artifacts/paths/outputs/".format(training_job.name),  # Use the correct job name
    name="bank_detection_model",  # Update the model name
    description="Model created for Churn",  # Update the description
    type="custom_model",  # Specify the type of the asset
)

# Create or update the model in the Azure ML workspace
ml_client.models.create_or_update(model)

Model({'job_name': 'polite_island_dxt1kxqhxs', 'intellectual_property': None, 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'bank_detection_model', 'description': 'Model created for Churn', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/6c23abb4-31cd-4bb7-bd34-4f4d970f93ea/resourceGroups/gl-oct24-pro/providers/Microsoft.MachineLearningServices/workspaces/gl-oct24-pro/models/bank_detection_model/versions/1', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/v01/code/Users/Vinicio_1726340028558', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x7f3d74c41330>, 'serialize': <msrest.serialization.Serializer object at 0x7f3d74c404f0>, 'version': '1', 'latest_version': None, 'path': 'azureml://subscriptions/6c23abb4-31cd-4bb7-bd34-4f4d970f93ea/resourceGroups/gl-oct24-pro/workspaces/gl-oct24-pro/datastores/workspaceartifactstore/paths/ExperimentRun/d

## Deploy the model as an online endpoint

*After we register our model, we can deploy it as an online endpoint—that is, as a web service in the Azure cloud.*

To deploy a machine learning service, we typically need:

1. The model assets that we want to deploy. These assets include the model's file and metadata that we already registered in our training job.

2. Some code to run as a service. The code executes the model on a given input request (an entry script). This entry script receives data submitted to a deployed web service and passes it to the model. After the model processes the data, the script returns the model's response to the client. The script is specific to our model and must understand the data that the model expects and returns. When we use an MLFlow model, Azure Machine Learning automatically creates this script for us.

## Create a new online endpoint

*As a first step to deploying our model, we need to create our online endpoint. The endpoint name must be unique in the entire Azure region. For this notebook, we create a unique name using a universally unique identifier (UUID).*

In [21]:
import uuid

# Creating a unique name for the endpoint
online_endpoint_name = "tf-nn-endpoint-" + str(uuid.uuid4())[:8]

In [22]:
from azure.ai.ml.entities import (
    ManagedOnlineEndpoint,
    ManagedOnlineDeployment,
    Model,
    Environment,
)

# create an online endpoint
endpoint = ManagedOnlineEndpoint(
    name=online_endpoint_name,
    description="Classify bank churn using TensorFlow",
    auth_mode="key",
)

endpoint = ml_client.begin_create_or_update(endpoint).result()

print(f"Endpoint {endpoint.name} provisioning state: {endpoint.provisioning_state}")

Endpoint tf-nn-endpoint-4255c574 provisioning state: Succeeded


*Once we create the endpoint, we can retrieve it as follows:*

In [23]:
endpoint = ml_client.online_endpoints.get(name=online_endpoint_name)

print(
    f'Endpoint "{endpoint.name}" with provisioning state "{endpoint.provisioning_state}" is retrieved'
)

Endpoint "tf-nn-endpoint-4255c574" with provisioning state "Succeeded" is retrieved


## Create a Scoring Script

In [24]:
%%writefile score.py
import json
import numpy as np
import tensorflow as tf
import os

# Simulating the init() function to load the model
def init():
    global loaded_cnn
    model_path = os.path.join(
        os.getenv("AZUREML_MODEL_DIR"), "outputs/model.keras"
    )
    loaded_nn = tf.keras.models.load_model(model_path)

# Simulating the run() function to make predictions
def run(raw_data):
    # Load data and convert to numpy array
    data = np.array(json.loads(raw_data)["data"], dtype=np.float32)

    # Make predictions
    result = loaded_nn.predict(data)

    # Generate prediction results
    predictions = []
    for i, prediction in enumerate(result):
        if prediction <= 0.5:
            predictions.append(f'no_churn')
        else:
            predictions.append(f'churn')
    return predictions

Writing score.py


In [26]:
registered_model = ml_client.models.get(name="bank_detection_model", version=1)

## Deploy the model to the endpoint

After we've created the endpoint, we can deploy the model with the entry script. An endpoint can have multiple deployments. Using rules, the endpoint can then direct traffic to these deployments.

In the following code, we create a single deployment that handles 100% of the incoming traffic. We use an arbitrary color name (tff-blue) for the deployment. You could also use any other name such as tff-green or tff-red for the deployment. The code to deploy the model to the endpoint does the following:

1. deploys the model that we registered earlier;
2. scores the model, using the score.py file; and
3. uses the same curated environment (that we declared earlier) to perform inferencing.

In [27]:
model = registered_model

from azure.ai.ml.entities import CodeConfiguration

# create an online deployment.
blue_deployment = ManagedOnlineDeployment(
    name="tff-blue",
    endpoint_name=online_endpoint_name,
    model=model,
    code_configuration=CodeConfiguration(code="", scoring_script="score.py"),
    environment=curated_env_name,
    instance_type="Standard_NC4as_T4_v3",
    instance_count=1,
)

blue_deployment = ml_client.begin_create_or_update(blue_deployment).result()

Check: endpoint tf-nn-endpoint-4255c574 exists


## Clean up resources

*If we won't be using the endpoint, we should delete it to stop using the resource. Make sure no other deployments are using the endpoint before we delete it.*

In [ ]:
ml_client.online_endpoints.begin_delete(name=online_endpoint_name)